# 17 — Interrogate Non-Linearity

Verifies that LASSO feature selection (Notebook 15) did not permanently discard
non-linear signals, and characterises the non-linearity of the features XGBoost
(Notebook 16) actually uses.

## Sections

1. **Linearity trap audit table** — LASSO vs. MI rank divergence per feature/outcome
2. **PDP + ICE for top LASSO-selected features** — shape of marginal effects
3. **PDP for MI-rescued features** — validates whether non-linear pattern is real
4. **SHAP interaction values** — joint effects LASSO cannot detect (top-20 features)
5. **Monotonicity check** — SHAP sign vs. domain expectation for key features
6. **Direction concordance table** — LASSO coefficient sign vs. SHAP mean sign

## Required environment variables
```
ADLS_ACCOUNT_NAME
ADLS_CONTAINER   (default: 'data')
```

In [ ]:
import os
import json
import re
import tempfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.inspection import PartialDependenceDisplay

import xgboost as xgb
import shap

from azure.identity import DefaultAzureCredential
import adlfs

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 40)

## Configuration

In [ ]:
ADLS_ACCOUNT_NAME = os.environ["ADLS_ACCOUNT_NAME"]
ADLS_CONTAINER    = os.getenv("ADLS_CONTAINER", "data")

TRAIN_END_YEAR = 2018
VAL_END_YEAR   = 2021

OUTCOMES = [
    "civil_war_onset",
    "coup_attempt",
    "regime_backsliding",
    "mass_unrest_onset",
    "humanitarian_crisis_onset",
]

# Domain-theory expected sign for key features (positive = increases instability risk)
THEORY_SIGNS = {
    "gdp_pc":                    -1,  # richer → more stable
    "gdp_pc_lag1":               -1,
    "polity2":                   -1,  # more democratic → lower coup risk
    "polity2_lag1":              -1,
    "food_price_index":          +1,  # higher food prices → more unrest
    "food_price_index_lag1":     +1,
    "regime_durability":         -1,  # older regime → lower transition risk
    "durable":                   -1,
    "polity_anocracy_flag":      +1,  # mid-range polity → higher risk
    "arch_irregular_exit_count": +1,  # past irregular exits → higher coup risk
    "arch_irregular_exit_count_lag1": +1,
    "vdem_v2x_libdem":           -1,  # more liberal democracy → lower backsliding risk
    "vdem_v2x_libdem_lag1":      -1,
    "ucdp_battle_deaths":        +1,  # ongoing violence → more onset risk
    "cnts_domestic_conflict_index": +1,
}

OUTCOME_COLS = [
    "civil_war_onset", "coup_attempt", "regime_backsliding",
    "mass_unrest_onset", "humanitarian_crisis_onset",
]
ID_COLS = ["iso3", "year"]

print(f"Outcomes       : {OUTCOMES}")
print(f"Theory signs   : {len(THEORY_SIGNS)} features")

## ADLS helpers and data loading

In [ ]:
credential = DefaultAzureCredential()
storage_options = {
    "account_name": ADLS_ACCOUNT_NAME,
    "credential":   credential,
}

_fs = adlfs.AzureBlobFileSystem(account_name=ADLS_ACCOUNT_NAME, credential=credential)

def _latest_date_dir(prefix: str) -> str | None:
    full = f"{ADLS_CONTAINER}/{prefix}"
    try:
        entries = _fs.ls(full, detail=False)
    except FileNotFoundError:
        return None
    dirs = sorted([e for e in entries if re.search(r'/\d{8}(/|$)', e)], reverse=True)
    return dirs[0] if dirs else None

def read_parquet_from_adls(path: str) -> pd.DataFrame:
    full = (
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}.dfs.core.windows.net/"
        + path.replace(f"{ADLS_CONTAINER}/", "", 1)
    )
    return pd.read_parquet(full, storage_options=storage_options)

def read_json_from_adls(path: str) -> dict | None:
    try:
        with _fs.open(path, "r") as f:
            return json.load(f)
    except FileNotFoundError:
        return None

def load_xgb_model(outcome: str) -> xgb.XGBClassifier | None:
    model_dir = _latest_date_dir(f"models")
    if not model_dir:
        return None
    files = _fs.glob(f"{model_dir}/{outcome}/model.json")
    if not files:
        return None
    with tempfile.NamedTemporaryFile(suffix=".json", delete=False) as tmp:
        with _fs.open(files[0], "rb") as src:
            tmp.write(src.read())
        tmp_path = tmp.name
    model = xgb.XGBClassifier()
    model.load_model(tmp_path)
    return model

# ── Load feature matrix and labels ───────────────────────────────────────────
_fm_dir = _latest_date_dir("processed/feature_matrix")

def _read_named(filename: str) -> pd.DataFrame | None:
    if not _fm_dir:
        return None
    files = _fs.glob(f"{_fm_dir}/{filename}")
    return read_parquet_from_adls(files[0]) if files else None

df_features = _read_named("feature_matrix.parquet")
df_labels   = _read_named("labels.parquet")

if df_features is not None:
    print(f"Feature matrix : {df_features.shape}")
    print(f"Labels         : {df_labels.shape if df_labels is not None else 'not loaded'}")
else:
    print("ERROR: could not load feature matrix")

## Section 1 — Linearity trap audit table

For each outcome, load the LASSO vs. MI audit parquet from Notebook 15 and display
the features where LASSO and MI rank most strongly diverge. Large rank gaps indicate
features where the linear assumption was most likely to have cost signal.

In [ ]:
audit_tables = {}
_fs_dir = _latest_date_dir("feature_selection")

for outcome in OUTCOMES:
    if _fs_dir is None:
        print(f"  {outcome}: no feature_selection partition found")
        continue
    files = _fs.glob(f"{_fs_dir}/audit_{outcome}.parquet")
    if not files:
        print(f"  {outcome}: audit parquet not found")
        continue
    audit = read_parquet_from_adls(files[0])
    audit_tables[outcome] = audit

    # Top divergers: rescued features + top-20 by |rank_gap|
    divergers = (
        audit[audit["rescued"] | (audit["rank_gap"].abs() > 20)]
        .sort_values("rank_gap", ascending=False)
        .head(25)
    )

    print(f"\n{'='*55}")
    print(f"Outcome: {outcome}")
    print(f"  LASSO non-zero : {audit['lasso_nonzero'].sum()}")
    print(f"  MI rescued     : {audit['rescued'].sum()}")
    print(f"\n  Top rank-gap divergers (LASSO vs. MI):")
    print(
        divergers[["feature", "lasso_coef", "mi_score",
                   "lasso_rank", "mi_rank", "rank_gap", "rescued"]]
        .to_string(index=False)
    )

## Sections 2 & 3 — PDP + ICE plots

**Section 2:** Top 10 LASSO-selected features per outcome.  
**Section 3:** MI-rescued features per outcome.

If the PDP line is curved or non-monotonic, the LASSO coefficient understated the
feature's importance. If ICE lines diverge widely from the PDP, there is heterogeneity
driven by interaction effects.

In [ ]:
def _prepare_test_X(outcome: str, feat_cols: list[str]) -> np.ndarray | None:
    """Subset feature matrix to test period and selected feature columns."""
    if df_features is None or df_labels is None:
        return None
    merged = df_features.merge(
        df_labels[["iso3", "year", outcome]], on=["iso3", "year"], how="inner"
    )
    merged = merged[merged["year"] > VAL_END_YEAR]
    merged = merged[merged[outcome].notna()]
    # Median impute (training medians unavailable here; use overall for display only)
    X = merged[feat_cols].fillna(merged[feat_cols].median())
    return X.values.astype(float)


def plot_pdp_ice(model, X_test, feat_cols, features_to_plot, outcome, section_label,
                 subsample: int = 200):
    """
    Render PDP + ICE for a list of feature names.

    Silently skips features not in feat_cols.
    """
    valid = [f for f in features_to_plot if f in feat_cols]
    if not valid:
        print(f"  {outcome} ({section_label}): no plottable features")
        return

    feat_indices = [feat_cols.index(f) for f in valid]

    # Subsample ICE curves
    rng = np.random.default_rng(42)
    sample_idx = rng.choice(len(X_test), size=min(subsample, len(X_test)), replace=False)
    X_sample = X_test[sample_idx]

    n_cols = min(4, len(valid))
    n_rows = (len(valid) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
    axes = np.array(axes).flatten()

    display = PartialDependenceDisplay.from_estimator(
        model, X_sample,
        features=feat_indices,
        feature_names=feat_cols,
        kind="both",       # PDP + ICE
        subsample=subsample,
        random_state=42,
        ax=axes[: len(valid)],
        n_jobs=-1,
    )
    for ax in axes[len(valid):]:
        ax.set_visible(False)

    fig.suptitle(f"PDP + ICE — {outcome} ({section_label})", fontsize=12)
    fig.tight_layout()
    plt.show()
    plt.close(fig)


_manifest_dir = _latest_date_dir("feature_selection")

for outcome in OUTCOMES:
    model = load_xgb_model(outcome)
    if model is None:
        print(f"  {outcome}: model not found — skipping PDP")
        continue

    # Load feature manifest
    manifest = None
    if _manifest_dir:
        files = _fs.glob(f"{_manifest_dir}/selected_{outcome}.json")
        if files:
            manifest = read_json_from_adls(files[0])

    if manifest is None:
        print(f"  {outcome}: no feature manifest — skipping PDP")
        continue

    feat_cols       = manifest.get("feature_names", [])
    lasso_features  = manifest.get("lasso_features", [])
    rescued_features = manifest.get("rescued_features", [])

    X_test = _prepare_test_X(outcome, feat_cols)
    if X_test is None or len(X_test) == 0:
        print(f"  {outcome}: no test data")
        continue

    # Section 2: top 10 LASSO features by |coef| (use audit if available)
    if outcome in audit_tables:
        top_lasso = (
            audit_tables[outcome]
            .loc[audit_tables[outcome]["lasso_nonzero"]]
            .nsmallest(10, "lasso_rank")["feature"]
            .tolist()
        )
    else:
        top_lasso = lasso_features[:10]

    plot_pdp_ice(model, X_test, feat_cols, top_lasso, outcome,
                 "top-10 LASSO features")

    # Section 3: MI-rescued features
    if rescued_features:
        plot_pdp_ice(model, X_test, feat_cols, rescued_features, outcome,
                     "MI-rescued features")
    else:
        print(f"  {outcome}: no rescued features to plot")

## Section 4 — SHAP interaction values

Heatmap of mean |SHAP interaction| for top-20 features. Off-diagonal cells with
large values represent joint effects that LASSO cannot detect.

In [ ]:
for outcome in OUTCOMES:
    model = load_xgb_model(outcome)
    if model is None:
        continue

    manifest = None
    if _manifest_dir:
        files = _fs.glob(f"{_manifest_dir}/selected_{outcome}.json")
        if files:
            manifest = read_json_from_adls(files[0])
    if manifest is None:
        continue

    feat_cols = manifest.get("feature_names", [])
    X_test    = _prepare_test_X(outcome, feat_cols)
    if X_test is None or len(X_test) == 0:
        continue

    # Limit to top-20 features by mean |SHAP| (interaction values are O(n²))
    explainer   = shap.TreeExplainer(model)
    X_sample    = X_test[:300]
    shap_vals   = explainer.shap_values(X_sample)
    mean_abs    = np.abs(shap_vals).mean(axis=0)
    top20_idx   = np.argsort(mean_abs)[::-1][:20]
    top20_names = [feat_cols[i] for i in top20_idx]
    X_top20     = X_sample[:, top20_idx]

    # Interaction values (expensive — capped at 200 rows)
    shap_ia = explainer.shap_interaction_values(X_top20[:200])
    mean_ia = np.abs(shap_ia).mean(axis=0)   # shape (20, 20)

    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(
        mean_ia,
        xticklabels=top20_names, yticklabels=top20_names,
        cmap="Blues", ax=ax,
        fmt=".3f", annot=len(top20_names) <= 15,
    )
    ax.set_title(f"Mean |SHAP interaction| — {outcome} (top-20 features)")
    plt.xticks(rotation=45, ha="right", fontsize=8)
    plt.yticks(fontsize=8)
    fig.tight_layout()
    plt.show()
    plt.close(fig)

    # Top 10 off-diagonal interaction pairs
    ia_flat = []
    for i in range(len(top20_names)):
        for j in range(i + 1, len(top20_names)):
            ia_flat.append((top20_names[i], top20_names[j], mean_ia[i, j]))
    top_pairs = sorted(ia_flat, key=lambda x: -x[2])[:10]
    print(f"\n{outcome} — top interaction pairs:")
    for f1, f2, val in top_pairs:
        print(f"  {f1:35s} × {f2:35s} = {val:.4f}")

## Sections 5 & 6 — Monotonicity check and direction concordance table

**Section 5:** For theory-signed features, compare expected effect direction against
mean SHAP sign. Flag disagreements.

**Section 6:** For all top-30 features per outcome, compare LASSO coefficient sign
against mean SHAP sign. >20% discordant features suggests revisiting λ threshold.

In [ ]:
concordance_tables = {}

for outcome in OUTCOMES:
    model = load_xgb_model(outcome)
    if model is None:
        continue

    manifest = None
    if _manifest_dir:
        files = _fs.glob(f"{_manifest_dir}/selected_{outcome}.json")
        if files:
            manifest = read_json_from_adls(files[0])
    if manifest is None:
        continue

    feat_cols = manifest.get("feature_names", [])
    X_test    = _prepare_test_X(outcome, feat_cols)
    if X_test is None or len(X_test) == 0:
        continue

    # Compute mean SHAP values on test set (capped at 500 rows)
    explainer = shap.TreeExplainer(model)
    X_s       = X_test[:500]
    shap_vals = explainer.shap_values(X_s)
    mean_shap = shap_vals.mean(axis=0)

    # ── Section 5: monotonicity check ─────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"Section 5 — Monotonicity check: {outcome}")
    print(f"{'Feature':<40} {'Theory':>8} {'SHAP sign':>10} {'OK?':>6}")
    print("-" * 68)

    for feat, theory_sign in THEORY_SIGNS.items():
        # Partial name match (e.g. 'polity2_lag1' matches 'polity2')
        idx = next(
            (i for i, f in enumerate(feat_cols)
             if f == feat or f.startswith(feat + "_")),
            None,
        )
        if idx is None:
            continue
        shap_sign = int(np.sign(mean_shap[idx]))
        ok = "✓" if shap_sign == theory_sign or shap_sign == 0 else "✗ FLAG"
        print(f"  {feat:<38} {'+' if theory_sign > 0 else '-':>8} "
              f"{'+'if shap_sign > 0 else ('-' if shap_sign < 0 else '0'):>10} {ok:>6}")

    # ── Section 6: direction concordance table (top 30 by |SHAP|) ─────────
    mean_abs  = np.abs(mean_shap)
    top30_idx = np.argsort(mean_abs)[::-1][:30]

    rows = []
    for i in top30_idx:
        feat = feat_cols[i]
        shap_sign = int(np.sign(mean_shap[i]))

        lasso_coef = 0.0
        if outcome in audit_tables and feat in audit_tables[outcome]["feature"].values:
            lasso_coef = float(
                audit_tables[outcome]
                .loc[audit_tables[outcome]["feature"] == feat, "lasso_coef"]
                .iloc[0]
            )
        lasso_sign = int(np.sign(lasso_coef))

        concordant = (lasso_sign == shap_sign) or lasso_sign == 0 or shap_sign == 0

        rows.append({
            "feature":        feat,
            "lasso_coef":     lasso_coef,
            "lasso_sign":     "+" if lasso_sign > 0 else ("-" if lasso_sign < 0 else "0"),
            "shap_mean":      float(mean_shap[i]),
            "shap_sign":      "+" if shap_sign > 0 else ("-" if shap_sign < 0 else "0"),
            "concordant":     concordant,
            "mean_abs_shap":  float(mean_abs[i]),
        })

    conc_df = pd.DataFrame(rows)
    concordance_tables[outcome] = conc_df

    n_disc = (~conc_df["concordant"]).sum()
    pct    = n_disc / len(conc_df)
    flag   = " ← REVIEW λ THRESHOLD" if pct > 0.20 else ""

    print(f"\nSection 6 — Direction concordance: {outcome}")
    print(f"  Discordant: {n_disc}/{len(conc_df)} ({pct:.0%}){flag}")
    print(conc_df[["feature", "lasso_sign", "shap_sign", "concordant", "mean_abs_shap"]]
          .to_string(index=False))